# Deep Q-Network v2 (Double DQN) GPU Training Pipeline for Bomberman RL
**Machine Learning Essentials — Final Project**

This notebook provides a complete Google Colab GPU training pipeline for **Agent 2 (DQN v2)**:
- Automatic CUDA acceleration detection & hardware diagnostics
- Double Deep Q-Network (DDQN) with 38-dimensional spatial feature representations
- Experience replay sampling & Polyak soft target network updates
- Reward shaping with true BFS potential differences
- Checkpoint persistence & model export

## 1. Setup Environment & Dependencies

In [ ]:
!pip install --upgrade numpy torch matplotlib scipy tqdm pygame

## 2. Clone Repository or Verify Directory

In [ ]:
import os
import sys

# Clone repository if running in a fresh Colab runtime
if not os.path.exists("agent_code"):
    !git clone https://github.com/MrMike24/Bomberman_rl.git
    %cd Bomberman_rl

os.makedirs("logs", exist_ok=True)
print("Working directory:", os.getcwd())
sys.path.insert(0, os.getcwd())


## 3. Hardware & CUDA Acceleration Detection

In [ ]:
import os
import torch

print("PyTorch Version:", torch.__version__)
cuda_available = torch.cuda.is_available()
print("CUDA Available:", cuda_available)

if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    device = torch.device("cpu")
    print("Using CPU mode (Single-threaded fallback)")

os.environ["TORCH_DEVICE"] = str(device)


## 4. Hyperparameter Configuration

In [ ]:
import os
import json

os.environ["AGENT_NAME"] = "project_dqn_v2"

HYPERPARAMETERS = {
    "agent": "project_dqn_v2",
    "architecture": "Double DQN (38 -> 128 -> 64 -> 6) with LayerNorm",
    "learning_rate": 0.0005,
    "gamma": 0.95,
    "epsilon_start": 1.0,
    "epsilon_decay": 0.996,
    "epsilon_min": 0.02,
    "replay_buffer_size": 50000,
    "batch_size": 64,
    "target_network_soft_tau": 0.005,
    "reward_profile": "profile_combative_survival",
    "optimizer": "AdamW",
    "loss_fn": "SmoothL1Loss (Huber)",
    "device": str(device)
}

print(json.dumps(HYPERPARAMETERS, indent=2))

os.makedirs("report_data", exist_ok=True)
with open("report_data/hyperparameters_v2.json", "w") as f:
    json.dump(HYPERPARAMETERS, f, indent=2)


## 5. Execute Double DQN v2 Training Curriculum

In [ ]:
import os
from train_dqn import train_dqn

os.makedirs("logs", exist_ok=True)
os.makedirs("report_data", exist_ok=True)

# Stage 1: Loot Crate Bombing & Navigation Curriculum (150 rounds)
print("=== Starting Stage 1: Loot-Crate Scenario ===")
train_dqn(
    n_rounds=150,
    scenario="loot-crate",
    opponents=[],
    reward_profile="profile_combative_survival",
    seed=42
)

# Stage 2: Classic Arena Competitive Combat Curriculum (350 rounds)
print("\n=== Starting Stage 2: Classic Arena Scenario ===")
train_dqn(
    n_rounds=350,
    scenario="classic",
    opponents=["peaceful_agent", "coin_collector_agent", "rule_based_agent"],
    reward_profile="profile_combative_survival",
    seed=101
)


## 6. Evaluate Trained Model Against Rule-Based Opponents

In [ ]:
from evaluate_agent import evaluate_agent

eval_results = evaluate_agent(
    agent_name="project_dqn_v2",
    opponents=["rule_based_agent", "rule_based_agent", "rule_based_agent"],
    scenario="classic",
    n_rounds=50,
    seed=42
)

print("\n================ TOURNAMENT EVALUATION (CLASSIC ARENA) ================")
for k, v in eval_results.items():
    print(f"{k:22s}: {v}")
print("=======================================================================")

## 7. Export Checkpoint & Download Model

In [ ]:
checkpoint_path = "agent_code/project_dqn_v2/model_data/model.pt"
if not os.path.exists(checkpoint_path) and os.path.exists("agent_code/project_dqn/model_data/model.pt"):
    import shutil
    os.makedirs("agent_code/project_dqn_v2/model_data", exist_ok=True)
    shutil.copy("agent_code/project_dqn/model_data/model.pt", checkpoint_path)

if os.path.exists(checkpoint_path):
    size_kb = os.path.getsize(checkpoint_path) / 1024.0
    print(f"Checkpoint confirmed at {checkpoint_path} ({size_kb:.2f} KB)")
    try:
        from google.colab import files
        files.download(checkpoint_path)
        print("Triggered Colab download dialog.")
    except ImportError:
        print("Running outside Colab; checkpoint saved locally.")
else:
    print("Error: Checkpoint file not found.")
